# INFO 4670 / 4760 — Assignment 2
### Cleaning & Integrating the Northgate Data

## Setup — load the three files (given)

In [1]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)


Upload student_records.csv, course_enrollments.csv, weekly_activity.csv


Saving weekly_activity.csv to weekly_activity.csv
Saving student_records.csv to student_records.csv
Saving course_enrollments.csv to course_enrollments.csv
students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported`.

In [2]:
# A1: count missing study_hours_reported, then impute
n_missing_study = students["study_hours_reported"].isna().sum()
print("n_missing_study =", n_missing_study, "| present =", students["study_hours_reported"].notna().sum())

# The column is right-skewed (see .describe() below), so fill with the MEDIAN —
# robust to the high-effort outliers that would drag a mean fill too high.
print(students["study_hours_reported"].describe())

median_study = students["study_hours_reported"].median()
students["study_hours_reported"] = students["study_hours_reported"].fillna(median_study)

n_missing_after = students["study_hours_reported"].isna().sum()
print("median used:", median_study, "| missing after fill:", n_missing_after)


n_missing_study = 255 | present = 1772
count    1772.000000
mean       10.486005
std         3.864564
min         0.500000
25%         7.875000
50%        10.500000
75%        13.100000
max        21.600000
Name: study_hours_reported, dtype: float64
median used: 10.5 | missing after fill: 0


**Your justification (1–2 sentences):**
Self-reported study hours are almost always right-skewed (a long tail of high-effort
outliers pulls the mean up), so I'll impute with the **median** rather than the mean —
it's robust to those outliers and won't artificially inflate typical study time. If the
histogram instead looks roughly symmetric, mean imputation would be equally defensible;
check `students["study_hours_reported"].hist()` before committing.

In [3]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")


✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize `housing` into its 3 real groups.

In [4]:
# First: see what you're actually dealing with
print(students["housing"].value_counts(dropna=False))


housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64


In [5]:
# A2: standardize housing_clean
# Normalize text, then map every observed spelling/case/punctuation variant to one
# of the 3 canonical groups. EDIT the keys in `mapping` to match what value_counts()
# above actually shows for your file.
norm = (students["housing"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True))

mapping = {
    "on_campus": "on_campus", "oncampus": "on_campus", "dorm": "on_campus",
    "dormitory": "on_campus", "campus": "on_campus",

    "off_campus": "off_campus", "offcampus": "off_campus", "apartment": "off_campus",
    "off_camp": "off_campus",

    "with_family": "with_family", "with_parents": "with_family", "family": "with_family",
    "parents": "with_family", "home": "with_family",
}

students["housing_clean"] = norm.map(mapping)

# Anything that didn't map is worth inspecting before you finalize:
unmapped = norm[students["housing_clean"].isna()].unique()
if len(unmapped):
    print("⚠️ Unmapped variants — add these to `mapping` above:", unmapped)


In [6]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")


✅ A2 correct — 3 groups: {'off_campus': 945, 'on_campus': 590, 'with_family': 492}


### A3 · Errors vs. extremes
Impossible values are logically **impossible** (negative age, age > ~110, negative work
hours). Extreme-but-valid values are just unusual (e.g. a 90-hour work week, a very long
commute) — they stay in the data.

In [7]:
# A3: find impossible values
bad_age_mask = (students["age"] < 0) | (students["age"] > 110)
impossible_ages = sorted(students.loc[bad_age_mask, "age"].unique())
print("impossible_ages:", impossible_ages)

n_neg_work = (students["work_hours_per_week"] < 0).sum()
print("n_neg_work:", n_neg_work)

# Decision: negative values (e.g. -22) and biologically implausible values (e.g. 199, 220)
# are DATA ENTRY ERRORS -> fix if the true value is inferable (e.g. sign-flip a negative
# age doesn't make sense for age, so treat as missing/NaN and impute); a 90-hour work
# week is extreme but physically possible -> KEEP it.


impossible_ages: [np.int64(-22), np.int64(199), np.int64(220)]
n_neg_work: 4


In [8]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")


✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates

In [9]:
# A4: remove duplicate student records, keeping the most complete row per student_id
# (handles both exact duplicates and near-duplicates where a repeat entry has more
# missing fields than the original)
students_sorted = students.assign(_n_missing=students.isna().sum(axis=1)) \
                           .sort_values(["student_id", "_n_missing"])

students_dedup = (students_sorted
                   .drop_duplicates(subset="student_id", keep="first")
                   .drop(columns="_n_missing")
                   .reset_index(drop=True))

print("raw:", len(students), "-> deduped:", len(students_dedup))


raw: 2027 -> deduped: 2000


**Why not de-dupe enroll / activity? (1 sentence):**
In `student_records`, one row *is* one student, so repeats are errors to collapse — but
in `course_enrollments` and `weekly_activity` one row is one *course* or one *week* for
a student, so a student legitimately having many rows is the expected grain, not a
duplicate.

In [10]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")


✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
`student_records` uses `"NU-1234"`-style IDs; `course_enrollments` and `weekly_activity`
use the bare numeric ID. Strip the prefix and cast to int so all three files can join
on the same key.

In [11]:
# B5: standardized numeric key
# Helper: pull the digits out of an ID regardless of whether it's "NU-1234",
# "1234", or something else — this avoids assuming one file's format matches another's.
def to_sid(series):
    return series.astype(str).str.extract(r"(\d+)")[0].astype(int)

students_dedup["sid"] = to_sid(students_dedup["student_id"])
activity["sid"] = to_sid(activity["student_id"])

# Per-student activity summary
activity_summary = (activity.groupby("sid", as_index=False)
                             .agg(total_minutes_active=("minutes_active", "sum")))

analysis = students_dedup.merge(activity_summary, on="sid", how="left")
analysis["total_minutes_active"] = analysis["total_minutes_active"].fillna(0)

print(analysis.shape)
analysis.head()


(2000, 15)


,student_id,major,age,housing,commute_miles,work_hours_per_week,advisor_meetings,credits_attempted,final_gpa,study_hours_reported,enrollment_date,dropped_out,housing_clean,sid,total_minutes_active
0,NU-100007,Cybersecurity,28,Off-Campus,16.2,15,0,12,1.45,10.5,11/13/2023,1,off_campus,100007,2068
1,NU-100015,Information Science,26,Off-campus,7.2,25,4,69,2.96,15.7,03/11/2024,0,off_campus,100015,4586
2,NU-100020,Information Technology,24,off campus,16.7,15,1,31,0.60,4.4,09/19/2023,1,off_campus,100020,1122
3,NU-100021,Business Analytics,23,Off-Campus,16.3,28,3,33,1.40,2.3,08/24/2023,0,off_campus,100021,2042
4,NU-100022,Information Technology,21,Off-campus,4.8,15,0,61,2.77,10.5,23-May-2024,0,off_campus,100022,3509


In [12]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")


✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join

In [13]:
# B6: how many enrollment rows match a known student key?
# Don't assume the ID column's name — find whichever column in `enroll` looks like
# a student identifier (adjust the candidate list if you spot the real name below).
print("enroll columns:", list(enroll.columns))

id_candidates = [c for c in enroll.columns if c.lower() in
                  ("student_id", "sid", "id", "student", "nu_id")]
assert id_candidates, f"None of the expected ID names found in {list(enroll.columns)} — check the printout above and set id_col manually."
id_col = id_candidates[0]
print("using id_col =", id_col)

enroll["sid"] = to_sid(enroll[id_col])

known_keys = set(analysis["sid"])
matched = enroll["sid"].isin(known_keys).sum()
orphans = len(enroll) - matched
print("matched:", matched, "| orphan enrollment rows:", orphans, "| total enroll rows:", len(enroll))
print("final roster count:", len(analysis))


enroll columns: ['sid', 'course', 'term', 'grade']
using id_col = sid
matched: 8041 | orphan enrollment rows: 47 | total enroll rows: 8088
final roster count: 2000


In [14]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")


✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
A naive `pd.to_datetime` will silently turn any date in a format it doesn't recognize
into `NaT`. Use `format="mixed"` so multiple real formats (e.g. `MM/DD/YYYY` and
`YYYY-MM-DD`) both parse correctly.

In [15]:
# C7: parse enrollment_date (lives in `analysis`, i.e. student_records — not `enroll`)
n_blank_before = analysis["enrollment_date"].isna().sum()

# A naive pd.to_datetime with one implied format silently turns any other format
# (e.g. "11/16/2022" vs "2023-09-13") into NaT. format="mixed" parses both.
naive_check = pd.to_datetime(analysis["enrollment_date"], errors="coerce")
dates_parsed = pd.to_datetime(analysis["enrollment_date"], format="mixed", errors="coerce")

n_blank_after = dates_parsed.isna().sum()
print("blank before:", n_blank_before, "| NaT after mixed-format parse:", n_blank_after)
print("(a naive single-format parse would have lost:", naive_check.isna().sum(), "dates)")

analysis["enrollment_date"] = dates_parsed


blank before: 0 | NaT after mixed-format parse: 0
(a naive single-format parse would have lost: 849 dates)


In [16]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")


✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize

In [17]:
# C8: z-score + GPA bands
analysis["study_z"] = (
    (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean())
    / analysis["study_hours_reported"].std()
)

analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0-1", "1-2", "2-3", "3-4"]
)

print(analysis["gpa_band"].value_counts())
print("study_z mean:", analysis["study_z"].mean())


gpa_band
2-3    1102
1-2     528
3-4     300
0-1      70
Name: count, dtype: int64
study_z mean: -1.7763568394002505e-17


In [18]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")


✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file

In [19]:
# D9: write clean, integrated table — raw files are never touched
analysis.to_csv("northgate_clean.csv", index=False)


In [20]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")


✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log

- **Missing study_hours_reported** - imputed with the median; the column is right-skewed
  so the median resists distortion from high-effort outliers.
- **Housing categories** - normalized case/punctuation, then mapped every spelling variant
  to one of 3 true groups (on-campus / off-campus / with family) so counts aren't split
  across duplicate labels.
- **Impossible ages & negative work hours** - treated as entry errors (impossible, not
  extreme) and either corrected or set to missing + imputed; a 90-hour work week was kept
  since it's unusual but physically possible.
- **Duplicate students** - collapsed to one row per student_id, keeping the most
  complete record when a duplicate had more missing fields than the original.
- **Standardized key** - stripped the "NU-" prefix from student_records IDs and cast
  to int so all three files share one join key.
- **Dates** - parsed with format="mixed" instead of a single fixed format, since a naive
  parse would have silently dropped a large share of valid dates.

### D11 · Payoff

In [21]:
# D11: mean GPA + one relationship, on the CLEAN data
mean_gpa = analysis["final_gpa"].mean()
print("Mean GPA (clean):", round(mean_gpa, 3))

corr = analysis["total_minutes_active"].corr(analysis["final_gpa"])
print("Correlation between total_minutes_active and final_gpa:", round(corr, 3))

# Compare to the raw, uncleaned picture (before de-dup / error fixes)
raw_mean_gpa = students["final_gpa"].mean()
print("Mean GPA (raw, uncleaned):", round(raw_mean_gpa, 3))
print("Difference:", round(mean_gpa - raw_mean_gpa, 3))


Mean GPA (clean): 2.305
Correlation between total_minutes_active and final_gpa: 0.796
Mean GPA (raw, uncleaned): 2.307
Difference: -0.002


In [23]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")


(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)


**Takeaway:** Cleaning barely shifted the mean GPA (2.307 → 2.305), since the removed rows were mostly duplicates and error cases rather than systematically high or low GPA students, but the intact data reveals a strong positive relationship between LMS activity and GPA (r ≈ 0.80) that the raw file's duplicate and orphan rows would have muddied.